In [1]:
# Ensure that we are using the correct host
import socket
try:
    assert "gpu" in socket.gethostname()
    print(f"Running on {socket.gethostname()}. All is good!")
except:
    raise RuntimeError(f"Be sure to run on GPU! You are currently running on {socket.gethostname()}")

Running on gpu13.storrs.hpc.uconn.edu. All is good!


In [2]:
import os
import pandas as pd
from pathlib import Path
import csv
from tqdm import tqdm
from collections import Counter

In [3]:
def analyze_file(filepath):
    """
    Analyze a single dataset file for row completeness.
    Skips first 9 columns and last 2 columns when counting completeness.
    Treats "555" and "888" as empty values.
    
    Returns:
        dict with filename, total_rows, filled_rows, incomplete_rows, empty_rows, empty_cell_percentage
    """
    try:
        # Try reading as TSV first (since your example is TSV)
        try:
            df = pd.read_csv(filepath, sep='\t', low_memory=False)
        except:
            # Fall back to regular CSV
            df = pd.read_csv(filepath, low_memory=False)
        
        total_rows = len(df)
        total_columns = len(df.columns)
        
        # Skip first 9 columns and last 2 columns
        # Only analyze the columns in between
        if total_columns > 11:  # Need at least 12 columns to have something to analyze
            df_analyze = df.iloc[:, 9:-2].copy()  # Skip first 9 and last 2
            analyzed_columns = len(df_analyze.columns)
        elif total_columns > 9:  # Has first 9 but not enough for last 2
            df_analyze = df.iloc[:, 9:].copy()  # Just skip first 9
            analyzed_columns = len(df_analyze.columns)
        else:
            # Not enough columns to skip, analyze nothing
            df_analyze = pd.DataFrame()
            analyzed_columns = 0
        
        if analyzed_columns > 0:
            # Replace "555" and "888" with NaN (treating them as empty)
            df_analyze = df_analyze.replace(['555', '888', 555, 888], pd.NA)
            
            # Count filled rows (all non-null in analyzed columns)
            filled_rows = df_analyze.dropna(how='any').shape[0]
            
            # Count completely empty rows (all null in analyzed columns)
            empty_rows = df_analyze.isna().all(axis=1).sum()
            
            # Count incomplete rows (some filled, some empty in analyzed columns)
            incomplete_rows = total_rows - filled_rows - empty_rows
            
            # Calculate percentage of empty cells
            total_cells = df_analyze.size  # total number of cells in analyzed columns
            empty_cells = df_analyze.isna().sum().sum()  # total number of empty cells
            empty_cell_percentage = (empty_cells / total_cells * 100) if total_cells > 0 else 0
        else:
            filled_rows = 0
            empty_rows = 0
            incomplete_rows = 0
            empty_cell_percentage = 0
        
        return {
            'filename': os.path.basename(filepath),
            'filepath': filepath,
            'total_rows': total_rows,
            'total_columns': total_columns,
            'analyzed_columns': analyzed_columns,
            'filled_rows': filled_rows,
            'incomplete_rows': incomplete_rows,
            'empty_rows': empty_rows,
            'empty_cell_percentage': round(empty_cell_percentage, 2),
            'status': 'success'
        }
        
    except Exception as e:
        return {
            'filename': os.path.basename(filepath),
            'filepath': filepath,
            'total_rows': 0,
            'total_columns': 0,
            'analyzed_columns': 0,
            'filled_rows': 0,
            'incomplete_rows': 0,
            'empty_rows': 0,
            'empty_cell_percentage': 0,
            'status': f'error: {str(e)}'
        }


def parse_data_dictionary(txt_filepath):
    """
    Parse a data dictionary TXT file and count variables by DataType.
    
    Returns:
        dict with counts for each data type found
    """
    try:
        # Read the TXT file (tab-separated)
        df = pd.read_csv(txt_filepath, sep='\t', low_memory=False)
        
        # Check if 'DataType' column exists
        if 'DataType' not in df.columns:
            return {
                'status': 'error: DataType column not found',
                'total_variables': len(df),
                'datatype_counts': {}
            }
        
        # Count occurrences of each data type
        datatype_counts = Counter(df['DataType'].dropna())
        
        return {
            'status': 'success',
            'total_variables': len(df),
            'datatype_counts': dict(datatype_counts)
        }
        
    except Exception as e:
        return {
            'status': f'error: {str(e)}',
            'total_variables': 0,
            'datatype_counts': {}
        }


def match_csv_to_txt(csv_filename, data_dict_dir):
    """
    Find the matching TXT file for a CSV file in the data dictionary directory.
    
    Returns:
        Path to matching TXT file or None
    """
    # Remove extension to get base name
    base_name = os.path.splitext(csv_filename)[0]
    
    # Try exact match with .txt extension
    txt_path = os.path.join(data_dict_dir, f"{base_name}.txt")
    if os.path.exists(txt_path):
        return txt_path
    
    # Try case-insensitive exact match
    try:
        all_txt_files = [f for f in os.listdir(data_dict_dir) if f.endswith('.txt')]
        base_lower = base_name.lower()
        
        for txt_file in all_txt_files:
            if txt_file.lower() == f"{base_lower}.txt":
                return os.path.join(data_dict_dir, txt_file)
    except:
        pass
    
    return None


def analyze_file_with_datadict(filepath, data_dict_dir):
    """
    Combined analysis: file completeness + data dictionary variable counts.
    
    Returns:
        dict with all analysis results including datatype counts
    """
    # First, do the standard file analysis
    result = analyze_file(filepath)
    
    # Then try to match and parse data dictionary
    filename = os.path.basename(filepath)
    txt_path = match_csv_to_txt(filename, data_dict_dir)
    
    if txt_path:
        dict_info = parse_data_dictionary(txt_path)
        
        # Add dictionary info to result
        result['data_dict_file'] = os.path.basename(txt_path)
        result['data_dict_status'] = dict_info['status']
        result['total_variables'] = dict_info['total_variables']
        
        # Add counts for each data type as separate columns
        datatype_counts = dict_info.get('datatype_counts', {})
        for dtype, count in datatype_counts.items():
            # Clean up the datatype name for column name
            clean_dtype = dtype.strip().replace(' ', '_')
            result[f'count_{clean_dtype}'] = count
    else:
        result['data_dict_file'] = 'NOT FOUND'
        result['data_dict_status'] = 'no match'
        result['total_variables'] = 0
    
    return result

In [4]:
data_dir = "/home/rif17002/honors_thesis/ABCD_files"
data_dict_dir = "/home/rif17002/honors_thesis/ABCD_data_dicts"
output_path = "/home/rif17002/honors_thesis/dataset_info"
output_file = "ksads_info.csv"

os.makedirs(output_path, exist_ok=True)

# Find all CSV and TSV files
file_patterns = ['*.csv', '*.tsv', '*.txt']
all_files = []

for pattern in file_patterns:
    all_files.extend(Path(data_dir).glob(pattern))

if not all_files:
    print(f"No CSV/TSV files found in {data_dir}")
else:
    print(f"Found {len(all_files)} files to analyze...")
    print(f"Data dictionary directory: {data_dict_dir}\n")
    
    # Analyze each file with progress bar
    results = []
    for filepath in tqdm(all_files, desc="Analyzing files", unit="file"):
        result = analyze_file_with_datadict(str(filepath), data_dict_dir)
        results.append(result)

Found 374 files to analyze...
Data dictionary directory: /home/rif17002/honors_thesis/ABCD_data_dicts



Analyzing files: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 374/374 [12:05<00:00,  1.94s/file]


In [5]:
if results:
    # Dynamically determine all fieldnames (including datatype counts)
    # Start with standard fields
    standard_fields = ['filename', 'filepath', 'total_rows', 'total_columns',
                      'analyzed_columns', 'filled_rows', 'incomplete_rows', 
                      'empty_rows', 'empty_cell_percentage', 'data_dict_file',
                      'data_dict_status', 'total_variables', 'status']
    
    # Find all datatype count columns
    all_keys = set()
    for result in results:
        all_keys.update(result.keys())
    
    datatype_fields = sorted([k for k in all_keys if k.startswith('count_')])
    
    # Combine into final fieldnames
    fieldnames = standard_fields + datatype_fields
    
    # Ensure all results have all fields (fill missing with 0 for counts)
    for result in results:
        for field in fieldnames:
            if field not in result:
                if field.startswith('count_'):
                    result[field] = 0
                elif field in ['data_dict_file', 'data_dict_status']:
                    result[field] = 'N/A'
                elif field == 'total_variables':
                    result[field] = 0
    
    # Write to CSV
    with open(os.path.join(output_path, output_file), 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(results)
    
    print(f"\n✓ Analysis complete! Results saved to: {output_file}")
    print(f"\nSummary:")
    print(f"  Total files analyzed: {len(results)}")
    print(f"  Successful: {sum(1 for r in results if r['status'] == 'success')}")
    print(f"  Errors: {sum(1 for r in results if r['status'] != 'success')}")
    print(f"  Data dicts found: {sum(1 for r in results if r.get('data_dict_file', 'NOT FOUND') != 'NOT FOUND')}")
    
    # Show datatype summary
    if datatype_fields:
        print(f"\nDataType Summary:")
        for dtype_col in datatype_fields:
            total = sum(r.get(dtype_col, 0) for r in results)
            if total > 0:
                dtype_name = dtype_col.replace('count_', '')
                print(f"  {dtype_name}: {total} variables")
else:
    print("No results to save.")


✓ Analysis complete! Results saved to: ksads_info.csv

Summary:
  Total files analyzed: 374
  Successful: 374
  Errors: 0
  Data dicts found: 0


In [6]:
# Path to your full analysis results
input_file = "/home/rif17002/honors_thesis/dataset_info/ksads_info.csv"
output_file = "/home/rif17002/honors_thesis/dataset_info/ksads_filtered.csv"

# Define which files you want to keep
files_to_keep = [
    'pdem02.txt',
    'abcd_cbcls01.txt',
    'abcd_cbcl01.txt',
    'abcd_ksads01.txt',
    'diff_emotion_reg_p01.txt',
    'opp_defiant_disorder_p01.txt',
    'depressive_disorders01.txt',
    'depressive_disorders_p01.txt',
    'disruptive_mood_dysreg01.txt',
    'disruptive_mood_dysreg_p01.txt',
]

# Read the full CSV and filter rows
filtered_results = []

with open(input_file, 'r') as f:
    reader = csv.DictReader(f)
    for row in reader:
        if row['filename'] in files_to_keep:
            filtered_results.append(row)

# Write filtered results to new CSV
if filtered_results:
    fieldnames = filtered_results[0].keys()
    
    with open(output_file, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(filtered_results)
    
    print(f"✓ Filtered results saved to: {output_file}")
    print(f"  Files included: {len(filtered_results)}")
    print("\nIncluded files:")
    for result in filtered_results:
        print(f"  - {result['filename']}")
else:
    print("No matching files found!")

✓ Filtered results saved to: /home/rif17002/honors_thesis/dataset_info/ksads_filtered.csv
  Files included: 9

Included files:
  - opp_defiant_disorder_p01.txt
  - diff_emotion_reg_p01.txt
  - disruptive_mood_dysreg_p01.txt
  - disruptive_mood_dysreg01.txt
  - depressive_disorders01.txt
  - pdem02.txt
  - depressive_disorders_p01.txt
  - abcd_cbcls01.txt
  - abcd_cbcl01.txt
